In [48]:
import _pickle as pickle
import gym
import numpy as np
import math
import os
import random
import torch
import torch.nn as nn
import torch.optim as optim
import torch.nn.functional as F

from envs import TFAugmentedFrozenLakeEnv

In [49]:
base_dir = "/home/bryanpu1/projects/thinking-as-control/EXPS/frozen_lake-v4-ppo-single_policy"
model_path = "n_thought_states_1-n_thought_acts_2.pt"
env_path = "n_thought_states_1-n_thought_acts_2-env.pkl"

In [50]:
policy = policy = torch.load(os.path.join(base_dir, model_path), weights_only=False)
env = pickle.load(open(os.path.join(base_dir, env_path), "rb"))

In [51]:
seed = 42

rng = np.random.RandomState(seed)
obs, _ = env.reset(rng.randint(0, 2 ** 10))
done = False

total_reward = 0.0
total_steps = 0
total_act_steps = 0

state_seq = [torch.tensor(np.hstack((obs["env"], obs["thought"])))]
action_seq = [torch.tensor([0])]
logit_seq = [torch.zeros((1, env.n_acts + env.n_thought_acts + 1))]
while not done:
    sseq = torch.stack(state_seq).unsqueeze(0)

    with torch.no_grad():
        logits, _ = policy(sseq[:, -1])
        # probs = F.softmax(logits, dim=-1)[0]
        # logging.info(probs)
        # dist = Categorical(probs)
        # action = dist.sample()
        action = torch.argmax(logits, dim=-1)

    next_obs, reward, terminated, truncated, _ = env.step(action.item())
    done = terminated or truncated

    if action > 0 and action < 5:
        total_act_steps += 1

    total_reward += reward
    total_steps += 1
    obs = next_obs
    logit_seq.append(logits)
    action_seq.append(action)
    state_seq.append(
        torch.tensor(np.hstack((obs["env"], obs["thought"])))
    )

In [52]:
total_reward, total_steps

(1.0, 16)

In [53]:
torch.cat((torch.vstack(state_seq)[:, [0]], torch.roll(torch.stack(action_seq), -1, dims=0)), dim=1)

tensor([[ 0.,  2.],
        [ 4.,  5.],
        [ 4.,  2.],
        [ 8.,  6.],
        [ 8.,  6.],
        [ 8.,  3.],
        [ 9.,  6.],
        [ 9.,  6.],
        [ 9.,  3.],
        [10.,  6.],
        [10.,  5.],
        [10.,  5.],
        [10.,  5.],
        [10.,  2.],
        [14.,  6.],
        [14.,  3.],
        [15.,  0.]], dtype=torch.float64)

In [54]:
torch.cat(logit_seq, dim=0)[:, 1:]

tensor([[ 0.0000,  0.0000,  0.0000,  0.0000,  0.0000,  0.0000],
        [ 2.4008,  3.4250, -5.0384,  2.3698,  1.2118,  0.8084],
        [ 2.5344,  1.7823, -3.8356,  0.5185,  2.7382,  2.5734],
        [ 1.8798,  4.4979, -6.1636, -0.2002,  2.8489,  2.4420],
        [ 2.6680,  0.1397, -2.6329, -1.3328,  4.2647,  4.3385],
        [ 2.2342, -3.0267,  2.7358, -1.9244,  2.9044,  3.1195],
        [ 2.0197, -4.6755,  4.3111, -1.8233,  2.5392,  2.9387],
        [ 2.7014, -0.2709, -2.3322, -1.7956,  4.6463,  4.7797],
        [ 2.0649, -2.9187,  2.4554, -2.7843,  3.4694,  3.6852],
        [ 1.5967, -4.2107,  4.0455, -3.1811,  3.0572,  3.4287],
        [ 2.7348, -0.6816, -2.0315, -2.2584,  5.0279,  5.2210],
        [ 2.5893, -1.8235,  0.3198, -2.3158,  4.4201,  4.4018],
        [ 2.0366,  0.0771, -2.1830, -2.3378,  4.5985,  4.4895],
        [ 1.0233,  2.7790, -5.0024, -2.7424,  4.9947,  4.3809],
        [ 0.1303,  5.0939, -6.8078, -3.0431,  5.0341,  4.0630],
        [ 2.8684, -2.3242, -0.8288, -4.1